## <a href="https://cursos.alura.com.br/course/langchain-desenvolva-agentes-inteligencia-artificial/task/161393?b2cUser=true"><b>Langchain Agentes - Criação da Ferramenta parte 1</b></a><br/>

<b>Objetivo:</b> Criação da ferramenta, que será chamada por um agente, capaz de extrair dados da Ana

In [1]:
#%pip install -r requirements.txt

In [2]:
from pydantic import BaseModel, Field

class ExtratorDeEstudante(BaseModel):
    estudante: str = Field(description="Nome do estudante informado, sempre em letras minúsculas. Exemplo: joão, carla, joana")

### <b>CRIAÇÃO DE FERRAMENTAS</b>
Que ferramentas eu tenho disponíveis para se obter os dados da Ana ?

<b>1) Criação da Ferramenta DadosDeEstudante</b>
<ul><li> A classe deve estender de BaseTool</li></ul>

In [3]:
from os import getenv
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import BaseTool
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.globals import set_debug

set_debug(True)

load_dotenv()

class DadosDeEstudante(BaseTool):
    
    # TODA FERRAMENTA PRECISA TER ESSES ATRIBUTOS
    name: str = "dados_de_estudante" # Nome da ferramenta
    description : str = """ 
                            Essa ferramenta extrai o histórico e preferências de um estudante, de acordo com o seu histórico.
                        """ # Descrição da ferramenta 
    
    # CONTRATO DA FERRAMENTA - O QUE ELA FAZ
    def _run(self, input: str) -> str:
         
        # Perguntando para a llm
        llm = ChatOpenAI(
                            model="gpt-5-mini",
                            api_key=getenv("API_KEY")            
                        )
        
        parseador = JsonOutputParser(pydantic_object=ExtratorDeEstudante)    
        
        template = PromptTemplate(
                                    template = """ 
                                                    Você deve analisar a {input} e extrair o nome de estudante informado.
                                            
                                                    FORMATO DE SAIDA:
                                                    {formato_saida} 
                                                """,
                                    input_variables = ["input"],
                                    partial_variables = {"formato_saida": parseador.get_format_instructions()}
                                 )
        
        cadeia = template | llm | parseador
        
        resposta = cadeia.invoke({"input": input})
        
        return resposta['estudante']


<b>2) Executando a Ferramenta</b>

In [4]:
pergunta = "Quais são os dados da Ana?"

resposta = DadosDeEstudante().run(pergunta)

print(resposta)

[tool/start] [tool:dados_de_estudante] Entering Tool run with input:
"Quais são os dados da Ana?"
[chain/start] [tool:dados_de_estudante > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Quais são os dados da Ana?"
}
[chain/start] [tool:dados_de_estudante > chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "input": "Quais são os dados da Ana?"
}
[chain/end] [tool:dados_de_estudante > chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [tool:dados_de_estudante > chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human:  \n                                                    Você deve analisar a Quais são os dados da Ana? e extrair o nome de estudante informado.\n\n                                                    FORMATO DE SAIDA:\n                                                    The output should be formatted as a JSON instance that co

A LLM tem que perceber que quando vier a pergunta "Quais são os dados da Ana?", ela tem que usar a ferramenta DadosDeEstudante. <br/>
Ela ainda não consegue fazer isso.